In [10]:
import pandas as pd
import numpy as np
import glob
import os
import cdsapi
import zipfile

In [11]:
def download_era5_data(cities_coords):
    cities_coords['lat_era5'] = np.round(cities_coords['latitude'] * 4) / 4
    cities_coords['lon_era5'] = np.round(cities_coords['longitude'] * 4) / 4

    client = cdsapi.Client()
    dataset = "reanalysis-era5-single-levels-timeseries"
    os.makedirs('era5', exist_ok=True)

    for index, row in cities_coords.iterrows():
        safe_name = str(row['name']).replace(" ", "_").replace("/", "-")
        target_file = f'era5/ERA5_{safe_name}-{row["code"]}.zip'
        
        if os.path.exists(target_file):
            continue
            
        print(f"[{index+1}/{len(cities_coords)}] Downloading: {row['name']}")
        
        request = {
            "variable": [
                "2m_temperature",
                "2m_dewpoint_temperature",
                "total_precipitation",
                "surface_solar_radiation_downwards",
                "total_sky_direct_solar_radiation_at_surface",
                "skin_temperature",                            
                "total_cloud_cover",
                "top_net_solar_radiation"                            
            ],
            "location": {
                "longitude": float(row['lon_era5']), 
                "latitude": float(row['lat_era5'])
            },
            "date": ["2003-01-01/2024-12-31"],
            "data_format": "csv"
        }

        client.retrieve(dataset, request).download(target_file)

In [12]:
def read_era5_data(input_folder):
    zip_files = glob.glob(os.path.join(input_folder, "ERA5_*.zip"))

    all_city_data = []
    for zip_path in zip_files:
        filename = os.path.basename(zip_path)
        city_name = filename.replace('ERA5_', '').replace('.zip', '').split('-')[0].replace('_', ' ')
        ibge_code = filename.replace('ERA5_', '').replace('.zip', '').split('-')[1]

        #print(f"Processing: {city_name}...")

        with zipfile.ZipFile(zip_path, 'r') as z:
            csv_names = [n for n in z.namelist() if n.endswith('.csv')]
            with z.open(csv_names[0]) as f:
                df_raw = pd.read_csv(f)
                df_raw['city'] = city_name
                df_raw['code'] = ibge_code
                all_city_data.append(df_raw)

    df_raw = pd.concat(all_city_data, ignore_index=True)
    df_raw['valid_time'] = pd.to_datetime(df_raw['valid_time'])
    df_raw.set_index('valid_time', inplace=True)
    
    return df_raw

In [13]:
def calculate_rh(t2m_c, d2m_k):
    d2m_c = d2m_k - 273.15
    es = 0.61078 * np.exp((17.27 * t2m_c) / (t2m_c + 237.3))
    e = 0.61078 * np.exp((17.27 * d2m_c) / (d2m_c + 237.3))
    rh = (e / es) * 100
    return np.clip(rh, 0, 100)

In [14]:
def process_era5_data(df_raw):

    # Filter for the critical Corn season in Mato Grosso (January to September)
    df_filtered = df_raw[(df_raw.index.month >= 1) & (df_raw.index.month <= 7)].copy()
    
    df_filtered['air_temperature_c'] = df_filtered['t2m'] - 273.15
    
    df_filtered['precipitation_mm'] = df_filtered['tp'] * 1000

    df_filtered['global_radiation_kj_m2'] = df_filtered['ssrd'] / 1000

    df_filtered['rh_pct'] = calculate_rh(df_filtered['air_temperature_c'], df_filtered['d2m'])

    df_filtered['year'] = df_filtered.index.year

    yearly_summary = df_filtered.groupby(['year', 'city', 'code']).agg(
        mean_temperature_c=('air_temperature_c', 'mean'),  
        max_temperature_c=('air_temperature_c', 'max'),    
        min_temperature_c=('air_temperature_c', 'min'),    
        total_rain_mm=('precipitation_mm', 'sum'),
        sum_global_radiation_kj_m2=('global_radiation_kj_m2', 'sum'),
        mean_relative_humidity_pct=('rh_pct', 'mean'),
        max_relative_humidity_pct=('rh_pct', 'max'),
        min_relative_humidity_pct=('rh_pct', 'min')
    ).reset_index().round(2)
    
    return yearly_summary

In [15]:
#download_era_5_data(pd.read_csv('ibge/cities_to_analize.csv'))

In [16]:
era5_raw = read_era5_data('era5')

In [19]:
df_era5_final = process_era5_data(era5_raw)

In [ ]:
%store df_era5_final

Stored 'df_era5_final' (DataFrame)
